In [ ]:
import numpy as np
import pandas as pd

import plotly.express as px
from statsmodels.tsa.seasonal import STL

from pathlib import Path
from datetime import datetime, timedelta

In [ ]:
data_dir = Path('data')
usage_data_fn  = data_dir / "TRUS_E_CUST_ICPCONS_202604_20260401_0000170112CK762.csv"
monthly_usage_data_fn = data_dir /  "TRUS_E_CUST_EIEP13B_202603_20260322_2054.csv"

In [ ]:
df = pd.read_csv(usage_data_fn)
df['start_dt'] = pd.to_datetime(df['start_dt'], format="%d/%m/%Y %H:%M")
df['end_dt'] = pd.to_datetime(df['end_dt'], format="%d/%m/%Y %H:%M")
display(df)

In [ ]:
monthly_data_df = pd.read_csv(monthly_usage_data_fn)

monthly_data_df.rename(
    columns={
        'Read period start date and time': 'start_dt',
        'Read period end date and time': 'end_dt'
    },
    inplace=True
)
monthly_data_df['start_dt'] = pd.to_datetime(monthly_data_df['start_dt'])
monthly_data_df['end_dt'] = pd.to_datetime(monthly_data_df['end_dt'])
display(monthly_data_df)
display(monthly_data_df.columns)

In [ ]:
monthly_data_df['calc'] = (
        monthly_data_df.apply(
        lambda x: df.loc[
            (df['start_dt'] >= x['start_dt'])
            & (df['end_dt'] < x['end_dt'])
        ]['units'].sum(),
        axis=1
    )
)
display(monthly_data_df.dtypes)

print(
    monthly_data_df['Active energy kWh'].sum(),
    monthly_data_df['calc'].sum()
)

In [ ]:
monthly_data_df['$'] = ( 
    monthly_data_df['Active energy kWh']*(monthly_data_df['$/kWh_anytime'] + monthly_data_df['$/kWh_EA_levy'])
    + monthly_data_df['daily_fixed_charge']*(monthly_data_df['end_dt'] - monthly_data_df['start_dt']).dt.days
)*1.15

px.line(monthly_data_df,x='start_dt',y='$').show()
px.line(monthly_data_df,x='start_dt', y='$/kWh_anytime').show()

In [ ]:
weekends = ["Saturday", "Sunday"]
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
office_weekdays = ["Monday", "Tuesday", "Thursday"]
wfh_weekdays = ["Wednesday", "Friday"]

date_fmt = "%d-%m-%Y"
japan_trip_start = datetime.strptime("09-09-2024", date_fmt)
japan_trip_end = datetime.strptime("22-09-2024", date_fmt)

christmas_break_2024_start = datetime.strptime("23-12-2024", date_fmt)
christmas_break_2024_end = datetime.strptime("30-12-2024", date_fmt)

canada_trip_start = datetime.strptime("18-10-2025", date_fmt)
canada_trip_end = datetime.strptime("09-11-2025", date_fmt)

christmas_break_2025_start = datetime.strptime("26-12-2025", date_fmt)
christmas_break_2025_end = datetime.strptime("03-01-2026", date_fmt)

extended_holidays = [
    (japan_trip_start, japan_trip_end),
    (christmas_break_2024_start, christmas_break_2024_end),
    (canada_trip_start, canada_trip_end),
    (christmas_break_2025_start, christmas_break_2025_end),
] 

In [ ]:
for (start, end) in extended_holidays:
    filter_idx = (df['start_dt'] >= start) & (df['start_dt'] <= end)
    week_before = (df['start_dt'] >= start - timedelta(days=7)) & (df['start_dt'] < start)
    week_after = (df['start_dt'] > end) & (df['start_dt'] <= end + timedelta(days=7))
    df.loc[filter_idx, 'units'] = df.loc[week_before | week_after, 'units'].mean() 

In [ ]:
df['time'] = df['start_dt'].dt.time
df['start_date'] = df['start_dt'].dt.date
df['start_month'] = df['start_date'].apply(lambda x: x.replace(day=1))
df['smoothed_units'] = df['units'].rolling(10).mean()

In [ ]:
daily_use = df.groupby('start_date')['units'].sum()
daily_use.replace(0, np.nan, inplace=True)

monthly_use = df.groupby('start_month')['units'].sum()
monthly_use.replace(0, np.nan, inplace=True)

In [ ]:
fig = px.line(df, x='start_dt', y='units', hover_data={"start_dt": "|%H:%M %A, %B %d %Y"})
fig.layout.yaxis.fixedrange = True
fig.show()

px.line(daily_use).show()

px.line(monthly_use).show()


In [ ]:
res = STL(daily_use, period=365, robust=True).fit()
trend = res.trend.rename('units').reset_index()
trend['series'] = 'trend'
observed = res.observed.rename('units').reset_index()
observed['series'] = 'observed'

px.line(pd.concat([trend, observed]), x='start_date', y='units', color='series').show()
px.line(res.seasonal).show()
px.scatter(res.resid).show()


In [ ]:
daily_use_ma_30_days = daily_use.rolling(30).mean().reset_index()
daily_use_ma_30_days['series'] = '30 day MA'

observed = daily_use.copy().reset_index()
observed['series'] = 'observation'

ma_comp = pd.concat([daily_use_ma_30_days, observed])
ma_comp['diff'] = observed['units'] - daily_use_ma_30_days['units']

px.line(ma_comp, x='start_date', y='units', color='series').show()
px.line(ma_comp,x='start_date', y='diff').show()
px.histogram(ma_comp,x='diff').show()

In [ ]:
daily_fft = np.fft.fft(daily_use)

px.scatter(x=np.real(daily_fft), y=np.imag(daily_fft)).show()

lp_daily_fft = np.copy(daily_fft)
hp_daily_fft = np.copy(daily_fft)

lp_daily_fft[np.abs(lp_daily_fft) > 1e3] = 0
hp_daily_fft[np.abs(hp_daily_fft) <= 1e3] = 0

lp_filtered_daily = np.real(np.fft.ifft(lp_daily_fft))
hp_filtered_daily = np.real(np.fft.ifft(hp_daily_fft))

lp_filtered_daily = pd.DataFrame(
    dict(
        units=lp_filtered_daily,
        datetime=daily_use.index,
    )
)

hp_filtered_daily = pd.DataFrame(
    dict(
        units=hp_filtered_daily,
        datetime=daily_use.index,
    )
)

px.line(lp_filtered_daily, x='datetime', y='units').show()
px.line(hp_filtered_daily, x='datetime', y='units').show()


hp_filtered_daily['start_month'] = pd.to_datetime(hp_filtered_daily['datetime']).apply(lambda x: x.replace(day=1)).dt.date

df_ = pd.concat([monthly_use, hp_filtered_daily.groupby('start_month')['units'].sum().rename('filtered')], axis=1)

px.line(df_, y=['units','filtered']).show()
display(df_.sum())


In [ ]:
weekday_quantiles_df = (
    df[
        df['start_dt'].dt.day_name().isin(weekdays)
    ]
    .groupby(
        by=['time', 'start_month']
    )
    .agg(
        lq=('units',lambda x: np.nanpercentile(x, 25)),
        median=('units',lambda x: np.nanpercentile(x, 50)),
        # mean=('units', 'mean'),
        uq=('units',lambda x: np.nanpercentile(x, 75)),
    )
    .reset_index()
)


weekday_quantiles_df = pd.melt(
    weekday_quantiles_df,
    id_vars=[
        'time',
        'start_month',
    ],
    value_vars=[
        'lq',
        'median',
        # 'mean',
        'uq'
    ],
    value_name='units',
    var_name='stat',
) 

display(weekday_quantiles_df)

max_val = weekday_quantiles_df['units'].max()

fig = px.line(
    weekday_quantiles_df, 
    x='time', 
    y='units', 
    color='stat',
    animation_frame='start_month', 
    animation_group='start_month',
    range_y=[0, max_val]
)
fig.update_layout(height=600)
fig.show()
fig.write_html('weekday_quantiles.html')

In [ ]:
weekday_stats_df = (
    df[
        df['start_dt'].dt.day_name().isin(wfh_weekdays)
    ]
    .groupby(
        by=['time', 'start_month']
    )
    .agg(
        mean=('units', 'mean'),
        std=('units', 'std'),
    )
    .reset_index()
)

fig = px.line(
    weekday_stats_df, 
    x='time', 
    y='std', 
    animation_frame='start_month', 
    animation_group='start_month',
    range_y=[0, weekday_stats_df['std'].max()]
)
fig.update_layout(height=600)
fig.show()

weekday_stats_df['mean+std'] = weekday_stats_df['mean'] + weekday_stats_df['std']
weekday_stats_df['mean-std'] = weekday_stats_df['mean'] - weekday_stats_df['std']


weekday_stats_df = pd.melt(
    weekday_stats_df,
    id_vars=[
        'time',
        'start_month',
    ],
    value_vars=[
        'mean-std',
        'mean',
        'mean+std',
    ],
    value_name='units',
    var_name='stat',
) 


max_val = 1.2*(weekday_stats_df['units']).max()
min_val = 1.2*(weekday_stats_df['units']).min()

fig = px.line(
    weekday_stats_df, 
    x='time', 
    y='units', 
    color='stat',
    animation_frame='start_month', 
    animation_group='start_month',
    range_y=[min_val, max_val]
)
fig.update_layout(height=600)
fig.show()